In [0]:
from utils.config import *

def set_secrets(SECRET_SCOPE, SECRET_KEY):
    """
    Sets up secerets for the Datalake2 Gen storage containers"""
    return     spark.conf.set(
        f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
    )

def build_location_path(dataset, layer):
    """
    Returns the location of conatiner for the specified dataset.
    """
    return BASE_PATH+CONFIG['datasets'][dataset]['paths'][layer]

def table_create_at_location(df, dataset, layer, table = None, mode="overwrite", schemaChange = False):
    """
    Creates or replaces a Delta table at the configured path.
    """

    path = build_location_path(dataset, layer)

    if layer == 'gold_base':
        if not table:
            raise ValueError("Gold tables must provide a table name")
        else:
            gold_table = CONFIG['datasets'][dataset]['tables']['gold'][table]
            path += '/'+gold_table
    else:
        path

    if mode == "overwrite":
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "false" if schemaChange == False else "true").save(path)

    elif mode == "append":
        df.write.format("delta").mode("append").option("mergeSchema", "false" if schemaChange == False else "true").save(path)

    else:
        raise ValueError(f"Unsupported mode: {mode}")

    return path

def select_from_delta(dataset, layer, table = None):
    """
    Returns a Spark DataFrame from the specified dataset and layer.
    """
    path = build_location_path(dataset, layer)

    if layer == 'gold_base':
        if not table:
            raise ValueError("Gold tables must provide a table name")
        else:
            gold_table = CONFIG['datasets'][dataset]['tables']['gold'][table]
            path += '/'+gold_table
    else:
        path
    return spark.read.format("delta").load(path)

def get_table_config(dataset,medallion_table):
    """
    Returns the table configuration for the specified medalion table. Write bronze,silver or gold.
    """
    return CONFIG['datasets'][dataset]['tables'][medallion_table]





In [0]:
def create_or_replace(df,table_name):
    """
    Create or replace a table.
    """
    df.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(table_name)

def append_data(df,table_name):
    """
    Append data to an existing table.
    """
    df.write.mode('append').saveAsTable(table_name)


In [0]:
def standardize_columns(df):
    """
    Standardize column names to lowercase and replace spaces with underscores.
    """
    return df.select(
        *[c.lower().replace(' ', '_') for c in df.columns]
    )

def add_meta_data_bronze(df):
    """
    Adds relevant meta_data columns to the df for bronze ingests.
    """
    return (
        df
        .select("*", "_metadata.file_name", "_metadata.file_modification_time", "_metadata.file_size", "_metadata.file_path")
        .withColumn("file_uploaded", F.current_timestamp())
    )

def regex_eplace_accross_columns(df,*args, current_substring, new_substring):
    """
    Replace substring values in rows for multiple columns in a DataFrame.
    """
    for column in args:
        df = df.withColumn(column, F.regexp_replace(F.col(column), current_substring, new_substring))
    return df


def select_columns(df, columns): #no real value in this function, but it's here for the sake of completeness
    """
    Select specific columns from a DataFrame.
    """
    return df.select(*columns)

def drop_columns(df, columns):
    """
    Drop columns from a DataFrame.
    """
    return df.drop(*columns)



In [0]:
def drop_nulls(df, column):
    """
    Drop rows with null values in a column.
    """
    return df.filter(df[column].isNotNull())

In [0]:
def validate_dataframe(
    df,
    null_checks=None,
    range_checks=None,
    duplicate_subset=None,
    mode="warn"
):
    """
    Validate Spark DataFrame using basic data quality rules.

    Parameters
    ----------
    df : Spark DataFrame
        Input dataframe to validate.

    null_checks : list[str], optional
        Columns that must not contain null values.

    range_checks : dict[str, tuple], optional
        Dictionary of numeric range validations.
        Example:
            {
                "departure_delay_minutes": (-1440, 1440)
            }

    duplicate_subset : list[str], optional
        Columns used to identify duplicate records.

    mode : str, default='warn'
        Validation behavior:
            - 'warn': return valid + invalid records
            - 'fail': raise exception if invalid rows found

    Returns
    -------
    tuple
        (valid_df, invalid_df)
    """

    null_checks = null_checks or []
    range_checks = range_checks or {}

    working_df = df.withColumn("error_reason", F.lit(None))

    # --------------------------------------------------
    # Null checks
    # --------------------------------------------------
    for col_name in null_checks:
        working_df = working_df.withColumn(
            "error_reason",
            F.when(
                F.col(col_name).isNull(),
                F.concat_ws(
                    "; ",
                    F.col("error_reason"),
                    F.lit(f"null in {col_name}")
                )
            ).otherwise(F.col("error_reason"))
        )

    # --------------------------------------------------
    # Range checks
    # --------------------------------------------------
    for col_name, (min_val, max_val) in range_checks.items():
        working_df = working_df.withColumn(
            "error_reason",
            F.when(
                (F.col(col_name) < min_val) | (F.col(col_name) > max_val),
                F.concat_ws(
                    "; ",
                    F.col("error_reason"),
                    F.lit(f"out of range: {col_name}")
                )
            ).otherwise(F.col("error_reason"))
        )

    # --------------------------------------------------
    # Duplicate checks
    # --------------------------------------------------
    if duplicate_subset:
        duplicate_rows = (
            working_df.groupBy(duplicate_subset)
            .count()
            .filter(F.col("count") > 1)
            .drop("count")
            .withColumn("is_duplicate", F.lit(1))
        )

        working_df = working_df.join(
            duplicate_rows,
            on=duplicate_subset,
            how="left"
        )

        working_df = working_df.withColumn(
            "error_reason",
            F.when(
                F.col("is_duplicate") == 1,
                F.concat_ws(
                    "; ",
                    F.col("error_reason"),
                    F.lit("duplicate record")
                )
            ).otherwise(F.col("error_reason"))
        ).drop("is_duplicate")

    # --------------------------------------------------
    # Split valid / invalid records
    # --------------------------------------------------
    invalid_df = working_df.filter(F.col("error_reason").isNotNull())
    count_invalid = invalid_df.count()

    valid_df = (
        working_df
        .filter(F.col("error_reason").isNull())
        .drop("error_reason")
    )

    # --------------------------------------------------
    # Fail mode behavior
    # --------------------------------------------------
    if invalid_df.limit(1).count() > 0:
        display(invalid_df)
        if mode == "fail":
            raise ValueError(
                "Data quality validation failed: invalid records detected."
            )
        elif mode == "drop":
            df = valid_df
            print(f"Data quality validation failed: {count_invalid} record(s) dropped.")
        elif mode == "warn":
            print(f"Data quality validation failed: {count_invalid} record(s) invalid.")
    else:
        print("Data quality validation passed.")
    return valid_df, invalid_df, df

In [0]:
def remove_existing_files(df, dataset,layer):
    """
    Removes existing files in the bronze table from df.
    """
    existing_files = select_from_delta(dataset,layer).select("file_path").distinct()

    path = build_location_path(dataset,layer)
    df = df.join(existing_files, on="file_path", how="left_anti")

    if df.count() == 0:
        raise ValueError(f"Found no new files in {path}.")
    return df

In [0]:
def merge_to_existing(df,dataset, pk, layer = 'silver_base', processed_ts = None):
    """
    Performs an upsert merge of the newrefined silver df into existing silver table.
    """
    if not pk:
        raise ValueError("Primary key(pk) must be provided for merge")
    
    if isinstance(pk, str):
        pk = [pk]

    path = build_location_path(dataset,layer)
    merge_condition = " AND ".join([f"target.{k} = updates.{k}" for k in pk])

    if DeltaTable.isDeltaTable(spark,path):
        print(f"Merging data into {path}")
        target = DeltaTable.forPath(spark,path)

        build = (
            target.alias("target")
            .merge(df.alias("updates"),merge_condition)
        )
        if processed_ts:
            build = build.whenMatchedUpdateAll(
           condition = f"updates.{processed_ts} >= target.{processed_ts}"
        )
        else:
            build = build.whenMatchedUpdateAll()
            
        build.whenNotMatchedInsertAll().execute()
    else:
        #First load
        print(f"First write of data into {path}")
        df.write.format("delta").mode("overwrite").save(path)
    return path